In [0]:
# Load Silver into a PySpark DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_path = "s3://stocks-silver-layer/processed/batch/"

stock_df = spark.read.parquet(silver_path)

print("Rows:", stock_df.count())
display(stock_df.limit(5))

In [0]:
# Create the window
window_spec = Window.partitionBy("ticker").orderBy("date")

In [0]:
# get previous closing price
stock_df = stock_df.withColumn(
    "previous_close",
    F.lag("close", 1).over(window_spec)
)

display(
    stock_df.select(
        "ticker",
        "date",
        "close",
        "previous_close"
    ).limit(20)
)

In [0]:
# Calculate daily return
stock_df = stock_df.withColumn(
    "daily_return",
    F.when(
        F.col("previous_close").isNotNull(),
        ((F.col("close") - F.col("previous_close")) / F.col("previous_close")) * 100
    )
)

display(
    stock_df.select(
        "ticker",
        "date",
        "close",
        "previous_close",
        "daily_return"
    ).limit(20)
)

In [0]:
# Create moving averages
window_7d = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
    .rowsBetween(-6, 0)
)

window_30d = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
    .rowsBetween(-29, 0)
)

In [0]:
# Calculate the averages
stock_df = stock_df.withColumn(
    "moving_avg_7d",
    F.avg("close").over(window_7d)
)

stock_df = stock_df.withColumn(
    "moving_avg_30d",
    F.avg("close").over(window_30d)
)
# View the result
display(
    stock_df.select(
        "ticker",
        "date",
        "close",
        "daily_return",
        "moving_avg_7d",
        "moving_avg_30d"
    ).limit(30)
)

In [0]:
# Daily price range
stock_df = stock_df.withColumn(
    "daily_range",
    F.col("high") - F.col("low")
)
# print the output
display(
    stock_df.select(
        "ticker",
        "date",
        "high",
        "low",
        "daily_range"
    ).limit(20)
)

In [0]:
# Calculate volatility -create the window
volatility_window = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
    .rowsBetween(-29, 0)
)
# Then calculate:

stock_df = stock_df.withColumn(
    "volatility_30d",
    F.stddev("daily_return").over(volatility_window)
)
# Now inspect it:

display(
    stock_df.select(
        "ticker",
        "date",
        "daily_return",
        "volatility_30d"
    ).limit(40)
)

In [0]:
# Let's inspect our transformed dataset
display(
    stock_df.select(
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume",
        "previous_close",
        "daily_return",
        "moving_avg_7d",
        "moving_avg_30d",
        "daily_range",
        "volatility_30d"
    ).limit(20)
)

In [0]:
# Create dim_stock
dim_stock = (
    stock_df
    .select("ticker")
    .distinct()
    .orderBy("ticker")
)

dim_stock = dim_stock.withColumn(
    "stock_key",
    F.row_number().over(Window.orderBy("ticker"))
)

dim_stock = dim_stock.select(
    "stock_key",
    "ticker"
)

display(dim_stock.limit(20))

In [0]:
# Create dim_date
# Now we'll create a Date dimension.
dim_date = (
    stock_df
    .select("date")
    .distinct()
)

# Then add the date attributes:

dim_date = (
    dim_date
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
)

# Then arrange the columns:

dim_date = dim_date.select(
    "date_key",
    "date",
    "year",
    "quarter",
    "month",
    "month_name",
    "day",
    "day_of_week"
).orderBy("date")

# Check it:

display(dim_date.limit(20))

In [0]:
# Create the fact table
# Now we'll connect our stock data to both dimensions.
fact_stock_prices = stock_df.join(
    dim_stock,
    on="ticker",
    how="left"
)

# Then add date_key:

fact_stock_prices = fact_stock_prices.withColumn(
    "date_key",
    F.date_format("date", "yyyyMMdd").cast("int")
)

# Now select the columns we actually want in the fact table:

fact_stock_prices = fact_stock_prices.select(
    "date_key",
    "stock_key",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
    "previous_close",
    "daily_return",
    "moving_avg_7d",
    "moving_avg_30d",
    "daily_range",
    "volatility_30d"
)

# Check it:

display(fact_stock_prices.limit(20))

In [0]:
# make sure our fact table didn't lose records during the join.
print("Silver rows:", stock_df.count())
print("Fact rows:", fact_stock_prices.count())

In [0]:
# Confirm your Gold bucket path
gold_base = "s3://stocks-gold-layer/"

In [0]:
# Write dim_stock

gold_base = "s3://stocks-gold-layer/"

dim_stock.write \
    .mode("overwrite") \
    .parquet(gold_base + "dim_stock/")

# Then verify:

display(
    spark.read.parquet(gold_base + "dim_stock/").limit(10)
)

In [0]:
# create dim_date
dim_date = (
    stock_df
    .select("date")
    .distinct()
    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.date_format("date", "EEEE"))
    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week"
    )
)

display(dim_date.limit(10))

In [0]:
# Write dim_date
dim_date.write \
    .mode("overwrite") \
    .parquet(gold_base + "dim_date/")

# Then verify:

display(
    spark.read.parquet(
        gold_base + "dim_date/"
    ).limit(10)
)

In [0]:
# Create fact_stock_prices
fact_stock_prices = (
    stock_df
    .join(
        dim_stock,
        on="ticker",
        how="left"
    )
    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .select(
        "date_key",
        "stock_key",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume",
        "previous_close",
        "daily_return",
        "moving_avg_7d",
        "moving_avg_30d",
        "daily_range",
        "volatility_30d"
    )
)

print("Fact rows:", fact_stock_prices.count())

display(fact_stock_prices.limit(10))

In [0]:
# Validate the fact table
print("Silver rows:", stock_df.count())
print("Fact rows:", fact_stock_prices.count())

print("Null stock keys:",
      fact_stock_prices.filter(F.col("stock_key").isNull()).count())

print("Null date keys:",
      fact_stock_prices.filter(F.col("date_key").isNull()).count())

In [0]:
# Write fact_stock_prices to Gold
fact_stock_prices.write \
    .mode("overwrite") \
    .parquet(gold_base + "fact_stock_prices/")

# Then verify:

gold_fact = spark.read.parquet(
    gold_base + "fact_stock_prices/"
)

print("Gold fact rows:", gold_fact.count())

display(gold_fact.limit(10))

In [0]:
# Final Gold check
gold_base = "s3://stocks-gold-layer/"

gold_dim_stock = spark.read.parquet(
    gold_base + "dim_stock/"
)

gold_dim_date = spark.read.parquet(
    gold_base + "dim_date/"
)

gold_fact = spark.read.parquet(
    gold_base + "fact_stock_prices/"
)

print("dim_stock rows:", gold_dim_stock.count())
print("dim_date rows:", gold_dim_date.count())
print("fact_stock_prices rows:", gold_fact.count())

print("\nGold tables loaded successfully!")

In [0]:
display(
    dbutils.fs.ls("s3://stocks-bronze-layer/raw/realtime_finnhub/")
)

In [0]:
raw = spark.read.text("s3://stocks-bronze-layer/raw/realtime_finnhub/AAPL_2026-08-10_13-25-13.json")

display(raw)

In [0]:
realtime_path = "s3://stocks-bronze-layer/raw/realtime_finnhub/"

realtime_df = spark.read.json(realtime_path)

display(realtime_df.limit(10))